In [1]:
import pandas as pd

In [2]:
FOLD = 2

In [4]:
flat_windowd_df = pd.read_csv(f"/scratch1/smaruj/genomic_flat_regions/flat_regions_tsv/fold{FOLD}_selected_genomic_windows_centered.tsv", sep="\t")

In [5]:
flat_windowd_df 

,chrom,start,end,fold,PearsonR,flat_start,flat_end,centered_start,centered_end,centered_flat_start,centered_flat_end
0,chr11,39598080,40908800,fold2,0.791830,250.0,379.0,39716864,41027584,192,320
1,chr11,42424320,43735040,fold2,0.833249,56.0,227.0,42188800,43499520,171,341
2,chr11,43407360,44718080,fold2,0.915895,167.0,359.0,43421696,44732416,160,352
3,chr11,44390400,45701120,fold2,0.836212,282.0,455.0,44619776,45930496,170,342
4,chr11,45701120,47011840,fold2,0.732358,331.0,455.0,45981696,47292416,194,318
5,chr11,55148544,56459264,fold2,0.910790,189.0,321.0,55146496,56457216,190,322
6,chr11,55803904,57114624,fold2,0.767461,288.0,455.0,56039424,57350144,173,339
7,chr14,10629120,11939840,fold2,0.731114,56.0,232.0,10399744,11710464,168,344
8,chr14,10956800,12267520,fold2,0.738000,316.0,455.0,11220992,12531712,187,325
9,chr14,12267520,13578240,fold2,0.800507,229.0,367.0,12353536,13664256,187,325


In [6]:
chromhmm_df = pd.read_csv("/home1/smaruj/mESC_mm10_3states_H3K27ac_9ac_9me3_chromHMM.bed", sep="\t", header=None)

In [7]:
chromhmm_df.columns = [
    "chrom",        # Column 1: Chromosome
    "start",        # Column 2: Start coordinate (0-based)
    "end",          # Column 3: End coordinate (exclusive)
    "state",        # Column 4: ChromHMM state (1, 2, or 3)
    "score",        # Column 5: Score (usually 0)
    "strand",       # Column 6: Strand (usually '.')
    "thickStart",   # Column 7: For browser display
    "thickEnd",     # Column 8: For browser display
    "rgb"           # Column 9: Color code for genome browser
]

In [8]:
state_map = {
    1: "active",
    2: "neutral",
    3: "repressive"
}
chromhmm_df["state_label"] = chromhmm_df["state"].map(state_map)

In [9]:
import bioframe as bf

In [10]:
flat_windowd_df = flat_windowd_df.rename(columns={"start": "og_start", "end": "og_end"})

In [11]:
cropping = 64
bin_size = 2048

In [12]:
# calculating genomic coordinates of flat regions
flat_windowd_df["start"] = flat_windowd_df["centered_start"] + (flat_windowd_df["centered_flat_start"] + cropping) * bin_size
flat_windowd_df["end"] = flat_windowd_df["centered_start"] + (flat_windowd_df["centered_flat_end"] + cropping) * bin_size

In [13]:
overlap_df = bf.overlap(
    flat_windowd_df,
    chromhmm_df,
    return_index=True,
    suffixes=("_query", "_chromhmm")
)

In [14]:
result = overlap_df[["chrom_query", 
                     "fold_query",
                     "PearsonR_query", 
                     "centered_start_query", 
                     "centered_end_query",
                     "centered_flat_start_query", 
                     "centered_flat_end_query", 
                     "state_chromhmm", 
                     "state_label_chromhmm"
]]

In [15]:
grouped = result.groupby(["chrom_query", 
                     "fold_query",
                     "PearsonR_query", 
                     "centered_start_query", 
                     "centered_end_query",
                     "centered_flat_start_query", 
                     "centered_flat_end_query", 
                     "state_chromhmm", 
                     "state_label_chromhmm"]).size().reset_index(name="count")

In [16]:
pivoted = grouped.pivot_table(
    index=["chrom_query", 
                     "fold_query",
                     "PearsonR_query", 
                     "centered_start_query", 
                     "centered_end_query",
                     "centered_flat_start_query", 
                     "centered_flat_end_query"],
    columns="state_label_chromhmm",
    values="count",
    fill_value=0
).reset_index()

In [17]:
pivoted.columns.name = None
pivoted = pivoted.rename(columns={
    "active": "active_count",
    "neutral": "neutral_count",
    "repressive": "repressive_count"
})

In [18]:
pivoted["total"] = pivoted[["active_count", "neutral_count", "repressive_count"]].sum(axis=1)

In [19]:
for label in ["active", "neutral", "repressive"]:
    pivoted[f"{label}_fraction"] = pivoted[f"{label}_count"] / pivoted["total"]

In [20]:
columns_to_remove = ["active_count", "neutral_count", "repressive_count", "total"]
pivoted.drop(columns=columns_to_remove, inplace=True)

In [21]:
pivoted.rename(columns={"chrom_query": "chrom", 
                        "fold_query": "fold",
                        "PearsonR_query": "PearsonR",
                        "centered_start_query": "centered_start",
                        "centered_end_query": "centered_end",
                        "centered_flat_start_query": "centered_flat_start",
                        "centered_flat_end_query": "centered_flat_end"}, inplace=True)

In [22]:
pivoted

,chrom,fold,PearsonR,centered_start,centered_end,centered_flat_start,centered_flat_end,active_fraction,neutral_fraction,repressive_fraction
0,chr11,fold2,0.732358,45981696,47292416,194,318,0.222222,0.555556,0.222222
1,chr11,fold2,0.767461,56039424,57350144,173,339,0.440000,0.520000,0.040000
2,chr11,fold2,0.791830,39716864,41027584,192,320,0.444444,0.555556,0.000000
3,chr11,fold2,0.833249,42188800,43499520,171,341,0.466667,0.533333,0.000000
4,chr11,fold2,0.836212,44619776,45930496,170,342,0.428571,0.523810,0.047619
5,chr11,fold2,0.910790,55146496,56457216,190,322,0.411765,0.529412,0.058824
6,chr11,fold2,0.915895,43421696,44732416,160,352,0.411765,0.529412,0.058824
7,chr14,fold2,0.731114,10399744,11710464,168,344,0.382353,0.500000,0.117647
8,chr14,fold2,0.738000,11220992,12531712,187,325,0.157895,0.526316,0.315789
9,chr14,fold2,0.763067,17168384,18479104,204,308,0.428571,0.571429,0.000000


In [ ]:
# pivoted.to_csv(f"/scratch1/smaruj/genomic_flat_loci/flat_regions_chrom_states_tsv/fold{FOLD}_selected_genomic_windows_centered_chrom_states.tsv", sep="\t", index=False)

In [ ]:
import matplotlib.pyplot as plt

# Sort by one of the fractions (optional)
pivoted_sorted = pivoted.sort_values("active_fraction", ascending=False)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))
bottoms = [0] * len(pivoted_sorted)

for label, color in zip(["active", "neutral", "repressive"], [None, None, None]):
    ax.bar(
        x=range(len(pivoted_sorted)),
        height=pivoted_sorted[f"{label}_fraction"],
        bottom=bottoms,
        label=label
    )
    bottoms = [sum(x) for x in zip(bottoms, pivoted_sorted[f"{label}_fraction"])]

ax.set_xlabel("Genomic Region")
ax.set_ylabel("Chromatin State Fraction")
ax.set_title("Chromatin State Composition per Flat Region")
ax.legend(title="Chromatin State")
ax.set_xticks([])  # Optional: hide x labels if not meaningful
plt.tight_layout()
plt.show()